# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors
This notebook provides a complete example for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described with a Croissant metadata schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's Croissant metadata and inspect general dataset properties with `mlcroissant`. The metadata provides high-level dataset details, including name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the root metadata object; do not subscript it
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
List available record sets in the dataset, including their `@id` and main fields. Per Croissant, all entities should be referenced by their `@id` field for clarity and reproducibility.

Below, we print all record set `@id`s and their associated field `@id`s. Then we preview the first few records of each record set for initial inspection.

In [ ]:
# List all record set @ids
record_sets = list(dataset.record_sets.keys())
if len(record_sets) == 0:
    print("No record sets found in this dataset.")
else:
    print("Available record sets and their fields:")
    for record_set_id in record_sets:
        record_set = dataset.record_sets[record_set_id]
        print(f"\nRecord set @id: {record_set_id}")
        if hasattr(record_set, 'fields'):
            field_ids = [field['@id'] if isinstance(field, dict) and '@id' in field else field for field in record_set.fields]
            print(f"  Fields (@id): {field_ids}")
        else:
            print("  (No fields found)")

    # Optionally, preview first record of each record set
    for record_set_id in record_sets:
        print(f"\nExample record from {record_set_id}:")
        records_iter = dataset.records(record_set=record_set_id)
        try:
            first_record = next(records_iter)
            print(first_record)
        except StopIteration:
            print("  (No records found in this record set)")

## 3. Data Extraction
For further analysis, load the main data table into a pandas DataFrame. All references use the `@id` from Croissant. Here, we assume the first record set is the main tabular record set.

Adjust the variable `main_record_set_id` as needed to target other record sets.

In [ ]:
# Get all record set @ids (if not already available)
record_sets = list(dataset.record_sets.keys())

# For this example, use the first record set as the main data table (adjust as needed)
main_record_set_id = record_sets[0]
print(f"Main record set: {main_record_set_id}")

# Load all records as a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
print(f"Columns in {main_record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Explore the tabular data for basic statistics. We'll pick a numeric field and group/categorize by another variable, referencing all by their `@id`.

Let's automatically select the first numeric field and a categorical/grouping field from the DataFrame, if available. We will filter records, normalize numeric data, and show grouped statistics.

In [ ]:
# Helper to choose numeric and group fields by data type/name heuristics
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
categorical_field_candidates = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]

if len(numeric_field_candidates) == 0:
    print("No numeric field found. Please set 'numeric_field_id' manually from the columns list above.")
    # Example: numeric_field_id = '<field_id>'
else:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")

    # Set a threshold for filtering
    threshold = df[numeric_field_id].quantile(0.5)  # use median for demonstration
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nRecords with {numeric_field_id} > {threshold:.2f} (filtered count: {len(filtered_df)}):")
    print(filtered_df.head())

    # Normalize the numeric field in the filtered set
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

    # Group by the first available categorical field (excluding any with large cardinality)
    group_field = None
    for field in categorical_field_candidates:
        if filtered_df[field].nunique() < filtered_df.shape[0] // 2:  # avoid near-uniques
            group_field = field
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped average of {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group (categorical) field found for grouping.")

## 5. Visualization
Plot distributions and relationships using the normalized numeric variable and (if found) the categorical grouping variable. All plots are labeled with the field's `@id`. Adjust plots below if different columns are of interest.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the normalized numeric field
if 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f'{numeric_field_id}_normalized'].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of Normalized {numeric_field_id}')
    plt.xlabel(f'{numeric_field_id}_normalized')
    plt.ylabel('Count')
    plt.show()

    # If group_field is available, boxplot by category
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(
            data=filtered_df,
            x=group_field,
            y=f'{numeric_field_id}_normalized',
            palette='Set2'
        )
        plt.title(f'Normalized {numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field selected, so no plot to display.")

## 6. Conclusion
This notebook demonstrated:  
- Loading FAIR² dataset metadata and records using `mlcroissant` from its Croissant schema URL.  
- How to extract, inspect, and process records by referencing entities via their Croissant `@id`.  
- Basic exploratory statistics and visualizations using pandas and seaborn.

Further analysis could include deeper clinical subsetting, survival modeling, or integration with additional cohort data, depending on research goals.